In [ ]:
# ========================================================================
# EN: This script performs two tasks:
#   1. Load a CSV file containing fire location suggestions, select necessary
#      columns (Fire_id, Lat, Lon, Date), clean missing values, and cast
#      the date column to datetime.
#   2. For a single fire (defined by hard‑coded number), open its Sentinel‑2
#      RGB image, let the user draw polygons around urban areas (villages,
#      towns), and save those polygons as a GeoPackage.
#
# BG: Този скрипт извършва две задачи:
#   1. Зарежда CSV файл с информации за пожари, избира нужните колони
#      (Fire_id, Lat, Lon, Date), премахва празните стойности и превръща
#      колоната с дата във формат datetime.
#   2. За конкретен пожар (зададен с твърдо число) отваря неговото
#      Sentinel‑2 RGB изображение, позволява на потребителя да очертае
#      полигони около населени места (села, градове) и записва тези
#      полигони като GeoPackage.
# ========================================================================

import pandas as pd

# ---- 1. LOAD AND CLEAN FIRE SUGGESTIONS CSV ---------------------------
# EN: Path to the CSV with fire suggestions (changed to new directory)
# BG: Път до CSV-файла с предложения за пожари (променен на новия път)
file_path = r'D:\data\master_thesis\input\fires_suggestion.csv'

# EN: Read the CSV file (not Excel, so read_csv is correct)
# BG: Прочитане на CSV файла (не е Excel, затова read_csv е правилно)
df = pd.read_csv(file_path)

# EN: Print all column names – use the output to verify column names below
# BG: Отпечатване на всички имена на колони – ползвайте изхода за проверка на имената по-долу
print("Columns in the file:")
print(df.columns.tolist())

# EN: Set the exact column names from the printed list (update if necessary)
# BG: Задаване на точните имена на колони от отпечатания списък (променете при нужда)
lat_col = 'Lat'       # EN: Latitude column   | BG: Колона за географска ширина
lon_col = 'Lon'       # EN: Longitude column  | BG: Колона за географска дължина
date_col = 'Date'     # EN: Date column       | BG: Колона за дата
fire_id = 'Fire_id'   # EN: Fire identifier   | BG: Идентификатор на пожар

# EN: Extract only the needed columns and create a copy
# BG: Извличане само на необходимите колони и създаване на копие
extracted_df = df[[fire_id, lat_col, lon_col, date_col]].copy()

# EN: Drop rows with any missing values in these critical columns
# BG: Премахване на редове с липсващи стойности в тези критични колони
extracted_df = extracted_df.dropna()

# EN: Convert the date column to datetime format (unparseable entries become NaT)
# BG: Конвертиране на колоната с дата във формат datetime (неразпознаваемите стойности стават NaT)
extracted_df[date_col] = pd.to_datetime(extracted_df[date_col], errors='coerce')

# EN: Final status message
# BG: Финално съобщение за състоянието
print(f"Done! Loaded and cleaned {len(extracted_df)} rows into 'extracted_df'.")
print("You can now use the DataFrame, e.g., extracted_df.head()")

# ---- 2. INTERACTIVE URBAN MASK CREATION FOR A SINGLE FIRE --------------
# RUN THIS IN A JUPYTER CELL (magic command to enable interactive plots)
# EN: Enable matplotlib interactive mode in Jupyter / Qt backend
# BG: Активиране на интерактивен режим на matplotlib в Jupyter / Qt бекенд
%matplotlib qt

import rasterio
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Polygon
import os
import numpy as np
import re

# ==========================================
# EN: HARDCODED PARAMETER – Change this number for a different fire
# BG: ТВЪРДО ЗАДАДЕН ПАРАМЕТЪР – Променете това число за друг пожар
# ==========================================
FIRE_NUMBER = 16
# ==========================================

# --- DIRECTORIES (updated to new paths) ---------------------------------
# EN: Base directory containing Sentinel‑2 RGB GeoTIFF images per fire
# BG: Основна директория, съдържаща Sentinel‑2 RGB GeoTIFF изображения за всеки пожар
base_image_dir = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures'

# EN: Output directory for GeoPackages with urban masks
# BG: Директория за изходните GeoPackage файлове с маски на населени места
output_dir = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures\geopackages'
os.makedirs(output_dir, exist_ok=True)

# EN: Sentinel-2 RGB bands: 4=Red, 3=Green, 2=Blue
# BG: Sentinel-2 RGB канали: 4=червен, 3=зелен, 2=син
RGB_BANDS = [4, 3, 2]

def process_fire():
    """
    EN: Interactive urban mask creation for a specific fire.
        Steps: find the image file, display RGB, let user draw polygons,
               convert pixel coordinates to map coordinates, save as GeoPackage.
    BG: Интерактивно създаване на маски за населени места за конкретен пожар.
        Стъпки: намиране на изображението, показване на RGB,
                чертане на полигони от потребителя, преобразуване на пикселни
                координати в картни, запис като GeoPackage.
    """
    # 1. EN: Find the first TIFF file whose name contains "fire{FIRE_NUMBER}"
    #    BG: Намиране на първия TIFF файл, чието име съдържа "fire{FIRE_NUMBER}"
    all_files = [f for f in os.listdir(base_image_dir) if f.endswith('.tif')]
    pattern = re.compile(f"fire{FIRE_NUMBER}[^0-9]")
    target_files = [f for f in all_files if pattern.search(f)]

    if not target_files:
        print(f"!!! Error (EN): Could not find image for Fire {FIRE_NUMBER} in directory. / Грешка: Няма изображение за Пожар {FIRE_NUMBER} в директорията.")
        return

    filename = target_files[0]
    fire_id = f"fire_{FIRE_NUMBER}"
    output_path = os.path.join(output_dir, f"{fire_id}_urban_masks.gpkg")

    full_image_path = os.path.join(base_image_dir, filename)
    print(f"--- ACTIVE SESSION: {fire_id.upper()} ---")
    print(f"EN: Creating urban masks for {fire_id} / BG: Създаване на маски за {fire_id}")
    print(f"Output will be saved to: {output_path}")

    with rasterio.open(full_image_path) as src:
        # 2. EN: Prepare RGB image for display with percentile stretch
        #    BG: Подготовка на RGB изображение за показване с персентилно разтягане
        try:
            img_data = src.read(RGB_BANDS)
        except:
            img_data = src.read([1, 2, 3])  # fallback if bands are indexed 1-3

        img_display = np.transpose(img_data, (1, 2, 0)).astype(float)
        for i in range(3):
            p5, p95 = np.percentile(img_display[:, :, i], (5, 95))
            img_display[:, :, i] = np.clip((img_display[:, :, i] - p5) / (p95 - p5), 0, 1)

        # 3. EN: Interactive drawing of urban polygons
        #    BG: Интерактивно чертане на полигони за населени места
        all_urban_geometries = []

        fig, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(img_display)
        ax.set_title(
            f"FIRE {FIRE_NUMBER} | URBAN MASK CREATION\n"
            "L‑Click: Add Points | R‑Click: Finish Polygon\n"
            "Draw villages and towns (close window when done)"
        )

        current_poly_pts = []
        polygons_drawn = 0

        def onclick(event):
            nonlocal current_poly_pts, polygons_drawn
            if event.button == 1:  # Left Click
                current_poly_pts.append((event.xdata, event.ydata))
                x, y = zip(*current_poly_pts)
                ax.plot(x, y, color='red', marker='o', markersize=4, linewidth=2)
                if len(current_poly_pts) > 1:
                    ax.plot([current_poly_pts[-2][0], current_poly_pts[-1][0]],
                            [current_poly_pts[-2][1], current_poly_pts[-1][1]],
                            color='red', linewidth=2)
                fig.canvas.draw()
                print(f"Point {len(current_poly_pts)}: ({event.xdata:.2f}, {event.ydata:.2f})")

            elif event.button == 3 and len(current_poly_pts) > 2:  # Right Click
                # EN: Close the polygon and convert pixel coordinates to map coordinates
                # BG: Затваряне на полигона и конвертиране на пикселни координати в картни
                ax.plot([current_poly_pts[-1][0], current_poly_pts[0][0]],
                        [current_poly_pts[-1][1], current_poly_pts[0][1]],
                        color='magenta', lw=3)

                try:
                    map_pts = []
                    for x, y in current_poly_pts:
                        col, row = int(x), int(y)
                        if 0 <= row < src.height and 0 <= col < src.width:
                            map_x, map_y = src.xy(row, col)
                            map_pts.append((map_x, map_y))
                        else:
                            print(f"Warning: Point ({x:.2f}, {y:.2f}) is outside image bounds / Точката е извън границите на изображението")

                    if len(map_pts) > 2:
                        polygon = Polygon(map_pts)
                        all_urban_geometries.append(polygon)
                        polygons_drawn += 1
                        print(f"✓ Created polygon {polygons_drawn} with {len(map_pts)} points / Създаден полигон {polygons_drawn} с {len(map_pts)} точки")
                        current_poly_pts = []
                        fig.canvas.draw()
                    else:
                        print("Error: Not enough valid points for polygon / Грешка: Недостатъчно валидни точки за полигон")
                        current_poly_pts = []
                except Exception as e:
                    print(f"Error creating polygon / Грешка при създаване на полигон: {e}")
                    current_poly_pts = []

        fig.canvas.mpl_connect('button_press_event', onclick)
        plt.show(block=True)
        plt.close(fig)

        # 4. EN: Save all drawn polygons as a single GeoPackage
        #    BG: Записване на всички начертани полигони в един GeoPackage
        if all_urban_geometries:
            gdf = gpd.GeoDataFrame({
                'fire_id': [FIRE_NUMBER] * len(all_urban_geometries),
                'class': ['urban'] * len(all_urban_geometries),
                'type': ['village_town'] * len(all_urban_geometries),
                'polygon_id': list(range(1, len(all_urban_geometries) + 1))
            }, geometry=all_urban_geometries, crs=src.crs)

            gdf.to_file(output_path, driver="GPKG")
            print(f"\n✅ SUCCESS (EN) / УСПЕХ: Saved {len(all_urban_geometries)} urban polygons to:")
            print(f"   {output_path}")
            print(f"File contains / Файлът съдържа:")
            print(f"   - {len(all_urban_geometries)} urban mask(s) / маски за населени места")
            print(f"   - CRS: {src.crs}")
            print(f"   - Fire ID: {FIRE_NUMBER}")

            # EN: Summary of polygon sizes (in map units)
            # BG: Обобщение на размерите на полигоните (в единици на координатната система)
            print("\nPolygon Summary / Обобщение на полигоните:")
            for i, geom in enumerate(all_urban_geometries, 1):
                area = geom.area
                print(f"   Polygon {i}: {len(geom.exterior.coords)-1} vertices, Area: {area:.2f} sq units / върха, Площ: {area:.2f} кв. единици")
        else:
            print(f"\n⚠️  No urban polygons were created for Fire {FIRE_NUMBER} / Не бяха създадени полигони за Пожар {FIRE_NUMBER}")
            print("No GeoPackage file was created. / Не беше създаден GeoPackage файл.")

    print(f"\nCompleted processing for Fire {FIRE_NUMBER} / Приключена обработка на Пожар {FIRE_NUMBER}")

if __name__ == "__main__":
    process_fire()